# US wind fleet — installed capacity + growth trend

The US Wind Turbine Database (USWTDB) records every utility-scale
turbine in the country — 75k+ units, lat/lon, capacity, year online,
make and model. This notebook plots total installed nameplate capacity
by state and traces the growth of typical turbine size over time.

In [ ]:
%pip install --quiet 'sondio[geo]>=0.1.2,<0.2' matplotlib

import sondio
# sondio >= 0.1.2 reads your key from Colab Secrets (🔑 sidebar),
# Kaggle Secrets, SONDIO_API_KEY env var, or ~/.sondio/config — in that
# order. Set whichever fits your environment.
print(f"sondio {sondio.__version__}")

In [ ]:
# Full fleet — ~75k rows, ~750 pages at the default page size. Narrow
# with `min_year=2010` if you only want modern installs.
turbines = sondio.wind_turbines(all_pages=True)
print(f"{len(turbines):,} turbines")
turbines[["p_name", "t_state", "p_year", "t_cap", "t_manu", "t_model"]].head()

In [ ]:
# State totals — installed MW and turbine count. The USWTDB capacity
# field t_cap is nameplate kW, so divide by 1000 for MW.
state_totals = (
    turbines.assign(mw=turbines["t_cap"].fillna(0) / 1000.0)
    .groupby("t_state")
    .agg(turbines=("id", "size"), total_mw=("mw", "sum"))
    .sort_values("total_mw", ascending=False)
)
state_totals.head(10)

In [ ]:
# Bar chart: top 15 states by installed capacity.
import matplotlib.pyplot as plt

top = state_totals.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top.index, top["total_mw"], color="#2a6cad")
ax.set_xlabel("Installed capacity (MW)")
ax.set_title("US wind fleet — top 15 states by nameplate capacity")
ax.grid(alpha=0.2, axis="x")
for y, (state, row) in enumerate(top.iterrows()):
    ax.text(row["total_mw"], y, f"  {int(row['turbines']):,}", va="center", fontsize=9, color="#555")
plt.tight_layout(); plt.show()

In [ ]:
# Turbine-size trend: median nameplate capacity by commissioning year.
# Newer projects ship bigger rotors — this plots it directly.
by_year = (
    turbines.dropna(subset=["p_year", "t_cap"])
    .groupby("p_year")
    .agg(median_kw=("t_cap", "median"), installs=("id", "size"))
    .query("installs >= 50")
)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(by_year.index, by_year["median_kw"] / 1000.0, marker="o", color="#1a6e3d")
ax.set_xlabel("commissioning year"); ax.set_ylabel("median nameplate capacity (MW)")
ax.set_title("Turbines are getting bigger — median nameplate capacity over time")
ax.grid(alpha=0.2); plt.tight_layout(); plt.show()

## Next steps
- Filter with `bbox=(west, south, east, north)` to zoom a single region.
- Drill to a specific operator with `project_name="Roscoe"` or `manufacturer="GE Wind"`.
- Cross with `sondio.us.phmsa.pipeline_incidents(...)` and `sondio.earthquakes(...)` to build a multi-energy infrastructure map.